
#### PythonによるExcel操作自動化の基本

準備：APIリクエストの準備


In [ ]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd # エクセル操作はpandas必須！

# 環境変数の取得 "../.env" にしなくても参照できてる？？
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"


1. PythonプログラムでExcelファイルを読み込む

In [4]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')

# データフレームを表示して確認 先頭から5行表示するメソッド
df.head()

,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


2. ExcelデータをLLMが理解できる形式に変換

In [5]:
# 2. データをLLM用にテキスト形式に変換
# ※最近「RAG（検索拡張生成）」や「関数呼び出し（tool calling）」もトレンド

# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


3. OpenAI APIの呼び出し

In [7]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

この売上データをもとに、いくつかの重要な傾向や洞察を抽出できます。

### 1. カテゴリー別の売上分析
- **食品、衣服、日用品**の3つのカテゴリーがあります。各カテゴリーの売上を集計し、どのカテゴリが最も収益を上げているかを確認します。
- 例えば、食品と衣服はおそらく販売量が大きく、単価も異なりますので、総売上や利益を比較することで、どのカテゴリーに重点を置くべきかを評価できます。

### 2. 利益率の分析
- **利益**は、単価から原価を引いたものになります。各商品の利益を計算し、利益率（利益 ÷ 売上）を求めることで、どの商品が最も効率よく利益を生み出しているかを評価できます。
- 例えば、高単価ながらも販売数量の少ない商品と、低単価で多く売れている商品を比較し、成長戦略を検討できます。

### 3. 商品の売上トレンド
- **売上日**をもとに、月単位や週単位での売上トレンドを分析します。特定の期間に売上が多かったり、逆に落ち込んでいる場合、その理由を探ります。
- 例：特定の季節において売上が向上した場合、その時期にマーケティングキャンペーンを強化または、新商品を投入する戦略を考えることができます。

### 4. 人気商品と不人気商品の特定
- 各商品の売上数量をもとに、人気商品や不人気商品を特定します。人気商品には在庫を増やすなどの対策を、不人気商品についてはプロモーションを計画または早期の販売停止を検討します。

### 5. 単価と販売数量の相関分析
- 売上は単価と数量の積ですが、これらの関係を分析することで、価格が売上に与える影響を可視化します。
- 価格を下げて数量が増加するのか、新しい価格帯が消費者の興味を引くのかを理解することが重要です。

### 6. 顧客行動の理解
- 顧客の購入パターンを理解するため、特定の商品を購入する顧客の共通点（年齢、性別、購入頻度など）を分析します。この情報を基に、ターゲット層に合わせたマーケティング計画を立てます。

### 結論
上記の分析を通じて、企業は売上を最大化するための戦略を見つけ、市場での競争力を高めることができます。具体的なデータ分析を進めることで、より詳細なインサイトを得られます。


4. LLMからの応答を取得し、Pythonで結果を処理

In [10]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)

                                                   結果
0                   この売上データをもとに、いくつかの重要な傾向や洞察を抽出できます。
1                                                    
2                                  ### 1. カテゴリー別の売上分析
3   - **食品、衣服、日用品**の3つのカテゴリーがあります。各カテゴリーの売上を集計し、どの...
4   - 例えば、食品と衣服はおそらく販売量が大きく、単価も異なりますので、総売上や利益を比較する...
5                                                    
6                                       ### 2. 利益率の分析
7   - **利益**は、単価から原価を引いたものになります。各商品の利益を計算し、利益率（利益 ...
8   - 例えば、高単価ながらも販売数量の少ない商品と、低単価で多く売れている商品を比較し、成長戦...
9                                                    
10                                   ### 3. 商品の売上トレンド
11  - **売上日**をもとに、月単位や週単位での売上トレンドを分析します。特定の期間に売上が多...
12  - 例：特定の季節において売上が向上した場合、その時期にマーケティングキャンペーンを強化また...
13                                                   
14                               ### 4. 人気商品と不人気商品の特定
15  - 各商品の売上数量をもとに、人気商品や不人気商品を特定します。人気商品には在庫を増やすなど...
16                                                   
17                          

5. 結果をExcelファイルなどに書き戻す

In [11]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

★実務では下記ワークフロー化するとよい

In [1]:
# エクセル資料(売上表の例)の分析


# 必要なライブラリ、モジュールの取得
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# 環境変数の取得
load_dotenv()

# OpenAIクライアントを生成。envファイルのAPI_...の値を読んでる。
cliant = OpenAI(api_key=os.environ['API_KEY'])

# OpenAIで利用するモデル名を指定
MODEL_NAME= "gpt-4o-mini"


# ★ここから、ワークフロー
print("処理を開始します")

# 1.エクセルファイルの読み込み
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
                   
# 2.データをLLM用に文字列（テキスト形式）へ変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ：\n{sales_data_text}\nこの売上データの傾向を分析して"

# 3.OpenAI APIの呼び出し
## まず役割を設定
role= "あなたはマーケティング分野に精通したデータサイエンティストです。" \
"企業の成長をサポートするために、効果的なインサイトを提供します。"
## APIへリクエスト
response = cliant.chat.completions.create(
    model = MODEL_NAME,
    messages = [
        {"role":"system", "content":role},
        {"role":"user", "content": prompt_text}
    ],
)
#    ↑ここのカンマは無くてもＯＫだが構造のわかりやすさのため

# 4. 分析結果を受け取り、データフレームに変換する（表示の準備）
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])

# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")
#-------------------------------------


処理を開始します
Excelファイルに分析結果を保存しました。


In [1]:
import os
print(os.getcwd())


c:\Users\user\Downloads\llmdev\12_excel


In [ ]:
# 環境変数を取得して表示api_key = os.getenv("API_KEY")
import os 
from dotenv import load_dotenv
load_dotenv("../.env")
api_key = os.getenv("API_KEY")
if api_key:
    print(" API_KEY が読み込まれました！")
    print(f"API_KEY = {api_key[:5]}...（一部表示）")
else:
    print(" API_KEY が読み込まれていません！")

 API_KEY が読み込まれました！
API_KEY = sk-pr...（一部表示）
